In [ ]:
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import LineString
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense, Conv1D, MaxPooling1D, UpSampling1D,
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import mean_squared_error

from datetime import datetime

In [ ]:
# DATASET

PATH_TRAINING_A = '../data/preprocessing/normalized/normalized_local_original.npy'
lines_a_noisy = np.load(PATH_TRAINING_A)

train_norm, temp = train_test_split(lines_a_noisy, test_size=0.2, random_state=42)  # 20% for val+test
val_norm, test_norm = train_test_split(temp, test_size=0.5, random_state=42)  # split temp equally 10% for val and test each 

print(f'Train Shape {train_norm.shape}, Val Shape {val_norm.shape}, Test Shape {test_norm.shape}')

#VARIATION IS TOO LITTLE, GRADIENTS ARE VANISHING

In [ ]:
#AE ARCHITECTURE

timesteps = 64
features = 2
latent_dim = 32  #Adjust for tighter bottleneck

inputs = Input(shape=(timesteps, features))

encoded = LSTM(128, return_sequences=True)(inputs)
encoded = LSTM(64, return_sequences=True)(encoded)
encoded = LSTM(latent_dim)(encoded)

decoded = RepeatVector(timesteps)(encoded)
decoded = LSTM(64, return_sequences=True)(decoded)
decoded = LSTM(128, return_sequences=True)(decoded)
decoded = TimeDistributed(Dense(features))(decoded)

lstm_autoencoder = Model(inputs, decoded)
#autoencoder.summary()


In [ ]:
#TRAINING SETUP 

epochs = 50
batch_size = 32
loss = 'huber' #mse, mae

early_stop = EarlyStopping(
    monitor='val_loss',     # what to watch
    patience=20,            # epochs to wait for improvement, If noisy → increase to 30, If stable → reduce to 10–15
    restore_best_weights=True   # IMPORTANT
)

# metrics: things to be observed, loss: what is actually used for learning
lstm_autoencoder.compile(optimizer='adam', loss=loss, metrics=['accuracy','mse', 'mae', 'cosine_similarity'])

In [ ]:
#TRAINING

timestamp = datetime.now().strftime("%d%m_%H%M")

history = lstm_autoencoder.fit(
    train_norm, train_norm,
    validation_data=(val_norm, val_norm),
    epochs=epochs,
    batch_size=batch_size,
    shuffle=True,
    callbacks=[early_stop]
)


In [ ]:
# TRAINING HISTORY 

def plot_history(history, epochs, batch_size, train_data_noisy):
    plt.figure(figsize=(10, 5))
    plt.plot(history.history['loss'], label='Training Loss', color='#143642')

    if 'val_loss' in history.history:
        plt.plot(history.history['val_loss'], label='Validation Loss', color='#EC9A29')

    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.title(f'Training Loss LSTM AE {epochs} Epochs {batch_size} Batches and Training Data Shape: {train_data_noisy.shape}')
    plt.show()


plot_history(history, epochs, batch_size, lines_a_noisy)

In [ ]:
loss = 'mae' #'mae', 'cosine_similarity'

plt.figure(figsize=(10,5))
plt.plot(history.history[f'{loss}'], label=f'{loss}')
plt.xlabel('Epoch')
plt.ylabel(f'{loss}')
plt.title(f'{loss} Curve')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
lstm_autoencoder.save(f'../checkpoints/autoencoder/model/{batch_size}_batches_{epochs}_epochs_huber_local_{timestamp}.keras')

In [ ]:
# PREDICTION
reconstructed = lstm_autoencoder.predict(test_norm)

In [ ]:
# PLOT PREDICTION

for i in range(110,120):
    plt.figure(figsize=(6,5))
    plt.plot(test_norm[i,:,0], test_norm[i,:,1], color='#EC9A29', label='Original')
    plt.plot(reconstructed[i,:,0], reconstructed[i,:,1], color='#143642', label='Reconstructed')
    plt.legend()
    #plt.axis('equal')
    plt.grid(True)
    plt.title('River Line Reconstruction')
    plt.show()
